# Reading scanned McStas results

A scanned McStas result contains one HDF5 entry for each scan point. This notebook creates a small multi-entry NeXus file and shows how to select an entry with `mcstastox.Read`.

## Create example scan data

McStasScript can run each scan point independently. Until a high-level scan helper is available, the resulting `entry1` groups can be combined into one file. This preserves real event data from each simulation while giving McStasToX the same multi-entry layout as a scan.

In [ ]:
from pathlib import Path
import tempfile

import h5py

import mcstastox
from read_example import make_instrument

In [ ]:
def make_scan(scan_values, ncount=10_000):
    scan_root = Path(tempfile.mkdtemp(prefix="mcstastox_scan_"))
    scan_path = scan_root / "scan.h5"
    instrument = make_instrument(square=True, banana=False)

    with h5py.File(scan_path, "w") as scan_file:
        for entry_number, wavelength in enumerate(scan_values, start=1):
            run_root = scan_root / f"run_{entry_number}"
            run_root.mkdir()
            instrument.set_parameters(
                wavelength=wavelength,
                delta_wavelength=0.1,
            )
            instrument.settings(
                ncount=ncount,
                output_path=str(run_root),
                suppress_output=True,
                NeXus=True,
            )
            data = instrument.backengine()
            source_path = Path(data[0].original_data_location) / "mccode.h5"
            with h5py.File(source_path, "r") as source_file:
                source_file.copy(
                    "entry1",
                    scan_file,
                    name=f"entry{entry_number}",
                )

    return scan_path

In [ ]:
scan_values = [1.5, 2.0, 2.5]
scan_path = make_scan(scan_values)
scan_path

## Inspect the entries

The scan points are stored as `entry1`, `entry2`, and `entry3`. Entry numbers are one-based, matching the `entry_number` argument used by `mcstastox.Read`.

In [ ]:
with h5py.File(scan_path, "r") as scan_file:
    entry_names = sorted(
        scan_file,
        key=lambda name: int(name.removeprefix("entry")),
    )

entry_names

## Read the default entry

Without an explicit entry number, McStasToX reads `entry1`. The instrument parameters identify the scan point that was selected.

In [ ]:
with mcstastox.Read(scan_path.parent, filename=scan_path.name) as loaded_data:
    print(f"Number of entries: {loaded_data.get_number_of_entries()}")
    print(loaded_data.get_instrument_parameters())

## Select a scan point

Pass a one-based `entry_number` to select another scan point. All data access through the `Read` object then refers to that entry.

In [ ]:
selected_entry = 2
with mcstastox.Read(
    scan_path.parent,
    filename=scan_path.name,
    entry_number=selected_entry,
) as loaded_data:
    parameters = loaded_data.get_instrument_parameters()
    components = loaded_data.get_components()

assert parameters["wavelength"] == scan_values[selected_entry - 1]
print(f"Selected entry: {selected_entry}")
print(f"Wavelength: {parameters['wavelength']} angstrom")
print(f"Components: {components}")

The same selection applies when loading event data or exporting to another format. Use `get_number_of_entries()` to discover how many scan points are available before choosing an entry.